# 01 — Pipeline: GMM Events → SRI Edges → Graph

Runs the full data pipeline for one context and inspects each stage:
1. Load & filter logger data  
2. GMM event detection  
3. SRI association calculation  
4. Graph construction + metrics  

In [ ]:
import sys
sys.path.insert(0, '..')   # make project root importable

import pandas as pd
import matplotlib.pyplot as plt
import networkx as nx

from config import LOGGER_FILE, OUTPUT_ROOT
from pipeline.gmm_events import load_logger, filter_by_year, filter_by_plot, filter_by_period, detect_events
from pipeline.sri_construction import compute_sri, sri_to_edge_list
from pipeline.graph_construction import analyse_graph

# ── Choose a context ──────────────────────────────────────────────────────────
YEAR   = 2016
PLOT   = 'SPRA'
PERIOD = 'daytime'   # or 'nighttime'

## 1. Load & filter logger

In [ ]:
logger = load_logger(LOGGER_FILE)
print(f'Full logger: {len(logger):,} rows')

df = filter_by_year(logger, YEAR)
df = filter_by_plot(df, PLOT)
df = filter_by_period(df, PERIOD)
print(f'After filter ({YEAR}, {PLOT}, {PERIOD}): {len(df):,} rows')
df.head()

In [ ]:
# Detection time distribution
fig, ax = plt.subplots(figsize=(10, 3))
ax.hist(df['_sec'] / 3600, bins=60, color='steelblue', edgecolor='white')
ax.set_xlabel('Hour of day')
ax.set_ylabel('Detections')
ax.set_title(f'Detection time distribution — {YEAR} {PLOT} {PERIOD}')
plt.tight_layout()
plt.show()

## 2. GMM event detection

In [ ]:
events, membership = detect_events(df)
print(f'Events detected:   {len(events):,}')
print(f'Membership rows:   {len(membership):,}')
print(f'Unique birds:      {membership["bird_id"].nunique()}')
events.head(10)

In [ ]:
# Event duration distribution
events['start_dt'] = pd.to_datetime(events['start_date'] + ' ' + events['start_time'])
events['end_dt']   = pd.to_datetime(events['end_date']   + ' ' + events['end_time'])
events['duration_min'] = (events['end_dt'] - events['start_dt']).dt.total_seconds() / 60

fig, axes = plt.subplots(1, 2, figsize=(12, 3))
axes[0].hist(events['duration_min'], bins=40, color='coral')
axes[0].set_xlabel('Event duration (min)')
axes[0].set_ylabel('Count')
axes[0].set_title('Event durations')

axes[1].hist(events['n_detections'], bins=20, color='mediumseagreen')
axes[1].set_xlabel('Detections per event')
axes[1].set_ylabel('Count')
axes[1].set_title('Event size')

plt.suptitle(f'{YEAR} {PLOT} {PERIOD}', y=1.02)
plt.tight_layout()
plt.show()

## 3. SRI calculation

In [ ]:
sri = compute_sri(events, membership, logger, plot=PLOT)
edges = sri_to_edge_list(sri)
print(f'SRI dyads: {len(sri):,}')
print(f'SRI range: {sri["SRI"].min():.4f} – {sri["SRI"].max():.4f}')
sri.head(10)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3))
ax.hist(sri['SRI'], bins=50, color='mediumpurple', edgecolor='white')
ax.set_xlabel('SRI')
ax.set_ylabel('Dyad count')
ax.set_title(f'SRI distribution — {YEAR} {PLOT} {PERIOD}')
plt.tight_layout()
plt.show()

## 4. Graph construction & metrics

In [ ]:
label   = f'{YEAR}_{PLOT}_{PERIOD}'
out_dir = OUTPUT_ROOT / label

results = analyse_graph(edges, output_dir=out_dir, prefix=label)

nodes = results['nodes']
stats = results['stats'].iloc[0]
print(f'Nodes: {int(stats["n_nodes"])}  Edges: {int(stats["n_edges"])}  Density: {stats["edge_density"]:.4f}')
print(f'Communities: {results["community_info"].iloc[0]["n_communities"]}  Modularity: {results["community_info"].iloc[0]["modularity"]:.4f}')
nodes.head()

In [ ]:
# Network position breakdown
if 'network_position' in nodes.columns:
    print(nodes['network_position'].value_counts())
if 'is_connector' in nodes.columns:
    print(f'\nConnector birds: {nodes["is_connector"].sum()}')

In [ ]:
# Quick network visualisation (spring layout, coloured by community)
G = nx.Graph()
for _, row in edges.iterrows():
    G.add_edge(row['id1'], row['id2'], weight=row['association'])

community_map = nodes.set_index('bird_id')['community'].to_dict() if 'community' in nodes.columns else {}
color_list    = [community_map.get(n, 0) for n in G.nodes()]

pos = nx.spring_layout(G, seed=42, weight='weight')
fig, ax = plt.subplots(figsize=(10, 8))
nx.draw_networkx(
    G, pos=pos, ax=ax,
    node_color=color_list, cmap='tab10',
    node_size=40, with_labels=False,
    edge_color='grey', alpha=0.7, width=0.5,
)
ax.set_title(f'Social network — {YEAR} {PLOT} {PERIOD}\n(colour = community)')
ax.axis('off')
plt.tight_layout()
plt.show()